In [1]:

import os
import re
import string
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

folder = "assignments"

documents = {}

for filename in os.listdir(folder):

    if filename.endswith(".txt"):

        path = os.path.join(folder, filename)

        with open(path, "r", encoding="utf-8") as file:
            documents[filename] = file.read()


def clean_text(text):

    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )
    text = re.sub(r"\s+", " ", text).strip()

    return text


cleaned_documents = {}

for filename, text in documents.items():
    cleaned_documents[filename] = clean_text(text)


document_names = list(cleaned_documents.keys())
document_texts = list(cleaned_documents.values())


vectorizer = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(
    document_texts
)


similarity_matrix = cosine_similarity(
    tfidf_matrix
)


similarity_df = pd.DataFrame(
    similarity_matrix,
    index=document_names,
    columns=document_names
)


print("=" * 60)
print("PLAGIARISM DETECTION REPORT")
print("=" * 60)

print("\nTotal Assignments:", len(document_names))

print("\nSimilarity Matrix:")
display(similarity_df.round(2))


threshold = 0.50

results = []

for i in range(len(document_names)):

    for j in range(i + 1, len(document_names)):

        document1 = document_names[i]
        document2 = document_names[j]

        score = similarity_matrix[i][j]

        percentage = score * 100

        if score >= threshold:
            status = "Possible Plagiarism"
        else:
            status = "Low Similarity"

        results.append({
            "Document 1": document1,
            "Document 2": document2,
            "Similarity (%)": round(percentage, 2),
            "Status": status
        })


report = pd.DataFrame(results)

report = report.sort_values(
    by="Similarity (%)",
    ascending=False
)

report = report.reset_index(drop=True)


print("\nRanked Similarity Report:")
display(report)


suspicious = report[
    report["Similarity (%)"] >= threshold * 100
]


print("\nPotentially Copied Documents:")

if len(suspicious) > 0:
    display(suspicious)
else:
    print("No potentially copied documents found.")


report.to_csv(
    "plagiarism_report.csv",
    index=False
)


print("\nReport saved as plagiarism_report.csv")


if len(report) > 0:

    highest = report.iloc[0]

    print("\nHighest Similarity Pair:")
    print(
        highest["Document 1"],
        "vs",
        highest["Document 2"]
    )

    print(
        "Similarity:",
        highest["Similarity (%)"],
        "%"
    )

    print(
        "Status:",
        highest["Status"]
    )

 

PLAGIARISM DETECTION REPORT

Total Assignments: 5

Similarity Matrix:


,student_A.txt,student_B.txt,student_C.txt,student_D.txt,student_E.txt
student_A.txt,1.00,0.47,0.09,0.01,0.03
student_B.txt,0.47,1.00,0.21,0.02,0.04
student_C.txt,0.09,0.21,1.00,0.10,0.08
student_D.txt,0.01,0.02,0.10,1.00,0.06
student_E.txt,0.03,0.04,0.08,0.06,1.00



Ranked Similarity Report:


,Document 1,Document 2,Similarity (%),Status
0,student_A.txt,student_B.txt,46.70,Low Similarity
1,student_B.txt,student_C.txt,21.14,Low Similarity
2,student_C.txt,student_D.txt,10.05,Low Similarity
3,student_A.txt,student_C.txt,9.44,Low Similarity
4,student_C.txt,student_E.txt,7.75,Low Similarity
5,student_D.txt,student_E.txt,5.57,Low Similarity
6,student_B.txt,student_E.txt,4.29,Low Similarity
7,student_A.txt,student_E.txt,3.47,Low Similarity
8,student_B.txt,student_D.txt,2.13,Low Similarity
9,student_A.txt,student_D.txt,0.95,Low Similarity



Potentially Copied Documents:
No potentially copied documents found.

Report saved as plagiarism_report.csv

Highest Similarity Pair:
student_A.txt vs student_B.txt
Similarity: 46.7 %
Status: Low Similarity
